# Notebook 09 — Non-Linearity Protocol Score (NLPS)

**This notebook aggregates all QNLEP metrics into a single composite score.**

```
NLPS = 0.35 × NLC
     + 0.25 × (1 - BPPI)    [inverted: lower BP proximity = better trainability]
     + 0.20 × TER
     + 0.20 × NSS
```

All four inputs are now **continuous reviewer-facing metrics**: continuous NLC,
smooth BPPI, continuous Lorenz TER, and bounded-skill NSS.

**Inputs:** JSON files produced by notebooks 03–08, plus optional `pqk_baseline_metrics.json` from **notebook 11** (comparison baselines — not in NLPS).  
**Outputs:** Score table, radar chart, component breakdown bar chart; optional NLC vs readout-variance scatter.

**Methods included (QNLEP registry):** data re-uploading, ancilla tracing,
ancilla re-uploading, strongly entangling layers, QCNN, quantum reservoir.

**Repro checklist:** run notebooks in the order given in `NEURIPS_NEXT_RUNS.md` (notebook **08** / TER before **09** when the composite includes TER). After **09**, report `results/nlps_composite_meta.json` together with `nlps_final_scores.json` whenever any method drops a component so readers see **effective** NLPS weights.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json, pathlib, sys

sys.path.insert(0, '..')
from src.qnlep_run_paths import protocol_results_dir

RESULTS_DIR = protocol_results_dir()


def get_score(d, method, default=np.nan):
    v = d.get(method, default)
    if v is None:
        return np.nan
    return v


## 1. Load metric scores from prior notebooks

In [ ]:
def load_json(name):
    path = RESULTS_DIR / name
    if path.exists():
        with open(path, encoding='utf-8') as f:
            return json.load(f)
    return {}

nlc_scores  = load_json('nlc_scores.json')    # from notebook 03
bppi_scores = load_json('bppi_scores.json')   # from notebook 04
ter_scores  = load_json('ter_scores.json')    # from notebook 08 (Lorenz TER)
nss_scores  = load_json('nss_scores.json')    # from notebook 07

# Optional: classical MLP baseline R² from notebook 06 (no NLC/BPPI — mark as N/A)
baseline    = load_json('baseline_scores.json')

print('Loaded scores:')
for name, d in [('NLC', nlc_scores), ('BPPI', bppi_scores),
                ('TER', ter_scores), ('NSS', nss_scores)]:
    print(f'  {name}: {d}')


## 1b. Load PQK-style baselines (notebook 11)

`pqk_baseline_metrics.json`: parameter counts and readout variance at random init — **not** part of NLPS; use for comparison figures.


In [ ]:
pqk_baselines = load_json("pqk_baseline_metrics.json")
print("PQK baselines:", pqk_baselines if pqk_baselines else "(empty — execute notebook 11)")


## 1c. QNLEP vs baseline scatter (optional)

Compare **NLC** (supervised frequency protocol) to **readout_var_z0** (random-parameter output spread). They need not correlate — that is the point of the protocol table in `NEURIPS_PROTOCOL.md`.


In [ ]:
if pqk_baselines and nlc_scores:
    common = sorted(set(nlc_scores) & set(pqk_baselines))
    xs = [get_score(nlc_scores, m) for m in common]
    ys = [pqk_baselines[m].get('readout_var_z0', float('nan')) for m in common]
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(xs, ys, s=80)
    for i, m in enumerate(common):
        ax.annotate(m, (xs[i], ys[i]), fontsize=7, xytext=(4, 4), textcoords='offset points')
    ax.set_xlabel('NLC (notebook 03)')
    ax.set_ylabel('readout_var_z0 (notebook 11)')
    ax.set_title('Protocol NLC vs PQK-style output spread')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    _figdir = pathlib.Path('../figures')
    _figdir.mkdir(parents=True, exist_ok=True)
    plt.savefig(_figdir / 'nlc_vs_readout_var.pdf', bbox_inches='tight')
    plt.show()
else:
    print('Skip scatter: need nlc_scores and pqk_baselines')


## 2. Assemble score matrix

If a method has not yet been run for a metric, its score defaults to `NaN`
and is excluded from the composite. This allows partial results during development.

In [ ]:
# Canonical row order (matches scripts/paper_export_multiseed_tables.py)
_METHOD_ORDER = (
    "strongly_entangling",
    "data_reuploading",
    "ancilla_tracing",
    "ancilla_reuploading",
    "qcnn",
    "quantum_reservoir",
)
_available = set(nlc_scores) | set(bppi_scores) | set(ter_scores) | set(nss_scores)
all_methods = [m for m in _METHOD_ORDER if m in _available]
for m in sorted(_available - set(all_methods)):
    all_methods.append(m)

FORMULA_WEIGHTS = {'NLC': 0.35, '1-BPPI': 0.25, 'TER': 0.20, 'NSS': 0.20}


def _components_for_method(m):
    nlc  = get_score(nlc_scores,  m)
    bppi = get_score(bppi_scores, m)
    ter  = get_score(ter_scores,  m)
    nss  = get_score(nss_scores,  m)
    return {
        'NLC': (nlc, 0.35),
        '1-BPPI': (1 - bppi if not np.isnan(bppi) else np.nan, 0.25),
        'TER': (ter, 0.20),
        'NSS': (nss, 0.20),
    }


score_matrix = {}
for m in all_methods:
    comps = _components_for_method(m)
    valid = {k: (v, w) for k, (v, w) in comps.items() if not np.isnan(v)}
    total_w = sum(w for _, w in valid.values())
    nlps = sum(v * w for v, w in valid.values()) / total_w if total_w > 0 else np.nan

    nlc, _ = comps['NLC']
    one_minus_bppi, _ = comps['1-BPPI']
    ter, _ = comps['TER']
    nss, _ = comps['NSS']
    bppi = np.nan if np.isnan(one_minus_bppi) else 1.0 - one_minus_bppi

    score_matrix[m] = {
        'NLC': nlc,
        'BPPI': bppi,
        '1-BPPI': one_minus_bppi,
        'TER': ter,
        'NSS': nss,
        'NLPS': nlps,
    }

print(f"\n{'Method':22s}  {'NLC':>6}  {'BPPI':>6}  {'TER':>6}  {'NSS':>6}  {'NLPS':>7}")
print('-' * 65)
for m in all_methods:
    s = score_matrix[m]
    def fmt(v):
        return f"{v:.3f}" if not np.isnan(v) else '  N/A'
    print(f"{m:22s}  {fmt(s['NLC'])}  {fmt(s['BPPI'])}  {fmt(s['TER'])}  {fmt(s['NSS'])}  {fmt(s['NLPS'])}")


def _nlps_effective_weights(m):
    comps = _components_for_method(m)
    valid = {k: (v, w) for k, (v, w) in comps.items() if not np.isnan(v)}
    tw = sum(w for _, w in valid.values())
    if tw <= 0:
        return {'valid_components': [], 'effective_weights': {}, 'weight_sum_used': 0.0}
    eff = {k: weight / tw for k, (_, weight) in valid.items()}
    return {
        'valid_components': list(valid.keys()),
        'effective_weights': eff,
        'weight_sum_used': tw,
    }


nlps_meta = {
    'formula_version': 'reviewer_proof_v2',
    'formula_weights': FORMULA_WEIGHTS,
    'component_files': {
        'NLC': 'nlc_scores.json',
        'BPPI': 'bppi_scores.json',
        'TER': 'ter_scores.json',
        'NSS': 'nss_scores.json',
    },
    'note': 'effective_weights renormalize to 1 over non-NaN NLPS components per method',
    'by_method': {m: _nlps_effective_weights(m) for m in all_methods},
}
with open(RESULTS_DIR / 'nlps_composite_meta.json', 'w', encoding='utf-8') as f:
    json.dump(nlps_meta, f, indent=2)
print('\nWrote results/nlps_composite_meta.json')

ter_available = any(not np.isnan(get_score(ter_scores, m)) for m in all_methods)
if not ter_available:
    print(
        'WARNING: TER missing for all methods - run notebook 08, then re-run this cell. '
        'See NEURIPS_NEXT_RUNS.md.',
    )


## 3. Visualizations

### 3a. Stacked bar chart — NLPS component breakdown

In [ ]:
WEIGHTS   = {'NLC': 0.35, '1-BPPI': 0.25, 'TER': 0.20, 'NSS': 0.20}
COLORS    = {'NLC': '#4C72B0', '1-BPPI': '#55A868', 'TER': '#C44E52', 'NSS': '#DD8452'}
LABELS    = {'NLC': 'NLC (×0.35)', '1-BPPI': '1−BPPI (×0.25)',
             'TER': 'TER (×0.20)', 'NSS': 'NSS (×0.20)'}

methods   = all_methods
x_pos     = np.arange(len(methods))
fig, ax   = plt.subplots(figsize=(7, 4))
bottom    = np.zeros(len(methods))

for metric, weight in WEIGHTS.items():
    values = np.array([score_matrix[m][metric] * weight
                       if not np.isnan(score_matrix[m][metric]) else 0.0
                       for m in methods])
    ax.bar(x_pos, values, bottom=bottom, color=COLORS[metric],
           edgecolor='white', lw=0.5, label=LABELS[metric])
    bottom += values

# NLPS total as text
for i, m in enumerate(methods):
    nlps = score_matrix[m]['NLPS']
    if not np.isnan(nlps):
        ax.text(i, bottom[i] + 0.01, f"{nlps:.2f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x_pos)
ax.set_xticklabels([m.replace('_', '\n') for m in methods], fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Weighted score contribution')
ax.set_title('NLPS — component breakdown per method')
ax.legend(fontsize=8, loc='upper right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
_out = pathlib.Path('../neurips2026/figures/nlps_stacked_bar.pdf')
_out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(_out, bbox_inches='tight')
plt.show()


### 3b. Radar chart — per-method profile

Shows the multi-dimensional profile of each method across all four QNLEP metrics.
Reveals trade-off structure: a method strong on NLC but weak on 1-BPPI has a
specific shape that a bar chart misses.

In [ ]:
RADAR_METRICS = ['NLC', '1-BPPI', 'TER', 'NSS']
N_AXES        = len(RADAR_METRICS)
angles        = np.linspace(0, 2 * np.pi, N_AXES, endpoint=False).tolist()
angles       += angles[:1]   # close the polygon

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for i, method_name in enumerate(methods):
    values = [score_matrix[method_name][m] for m in RADAR_METRICS]
    # Replace NaN with 0 for plotting
    values = [0.0 if np.isnan(v) else v for v in values]
    values += values[:1]

    color = plt.cm.tab10.colors[i]
    ax.plot(angles, values, 'o-', lw=2, color=color, label=method_name)
    ax.fill(angles, values, alpha=0.07, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(RADAR_METRICS, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.5', '0.75', '1.0'], fontsize=7)
ax.set_title('QNLEP radar — per-method profile', fontsize=11, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)
plt.tight_layout()
plt.savefig('../figures/nlps_radar.pdf', bbox_inches='tight')
plt.show()


### 3c. NLPS ranking bar chart with confidence intervals (placeholder)

In [ ]:
# Sort methods by NLPS
sorted_methods = sorted(
    [m for m in methods if not np.isnan(score_matrix[m]['NLPS'])],
    key=lambda m: score_matrix[m]['NLPS'], reverse=True
)
nlps_vals = [score_matrix[m]['NLPS'] for m in sorted_methods]

stats_path = RESULTS_DIR / 'nlps_final_scores_stats.json'
use_errorbars = stats_path.exists()
nlps_std = None
if use_errorbars:
    with open(stats_path, encoding='utf-8') as f:
        nlps_stats = json.load(f)
    nlps_std = [nlps_stats.get(m, {}).get('NLPS', {}).get('std', 0.0) for m in sorted_methods]

fig, ax = plt.subplots(figsize=(6, 3.5))
y_pos = np.arange(len(sorted_methods))
bar_kwargs = dict(color='steelblue', edgecolor='k')
if use_errorbars:
    ax.barh(y_pos, nlps_vals, xerr=nlps_std, error_kw=dict(capsize=4, elinewidth=1.2), **bar_kwargs)
else:
    ax.barh(y_pos, nlps_vals, **bar_kwargs)
ax.set_yticks(y_pos)
ax.set_yticklabels([m.replace('_', ' ') for m in sorted_methods], fontsize=9)
ax.set_xlabel('NLPS')
ax.set_xlim(0, 1.05)
ax.set_title('Non-Linearity Protocol Score ranking')
ax.axvline(0.5, color='grey', ls=':', lw=1)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/nlps_ranking.pdf', bbox_inches='tight')
plt.show()


### 3d. BPPI vs NLC scatter — expressiveness–trainability tradeoff

The central finding of the paper: if BPPI increases with NLC, the tradeoff is real;
if a method appears in the top-left (high NLC, low BPPI), it beats the tradeoff.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

for i, m in enumerate(methods):
    nlc  = score_matrix[m]['NLC']
    bppi = score_matrix[m]['BPPI']
    if np.isnan(nlc) or np.isnan(bppi):
        continue
    one_m_bppi = 1.0 - bppi
    ax.scatter(nlc, one_m_bppi, s=120, color=plt.cm.tab10.colors[i], zorder=3,
               label=m.replace('_', ' '), edgecolors='k', linewidths=0.7)

# Informal guideline stated in NEURIPS_PROTOCOL.md: BPPI < 0.3 as a ``safe'' regime
ax.axhline(0.7, color='orange', ls='--', lw=1.2,
           label='informal guideline (protocol doc): BPPI<0.3  ⇔  1−BPPI>0.7')

ax.set_xlabel('NLC (higher = more Fourier reach under the protocol budget)')
ax.set_ylabel('1 − BPPI (higher = healthier gradient scaling in n)')
ax.set_title('Expressiveness–trainability tradeoff')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/nlc_vs_bppi_scatter.pdf', bbox_inches='tight')
plt.show()


## 4. Export final score table

In [ ]:
# Save full score matrix for paper tables
export = {m: {k: (None if np.isnan(v) else round(v, 4))
              for k, v in s.items()}
          for m, s in score_matrix.items()}

with open(RESULTS_DIR / 'nlps_final_scores.json', 'w') as f:
    json.dump(export, f, indent=2)
print('Saved nlps_final_scores.json')

# Pretty-print for paper
print()
print(f"{'Method':22s}  {'NLC':>5}  {'BPPI':>5}  {'TER':>5}  {'NSS':>5}  {'NLPS':>6}")
print('─' * 60)
for m in sorted(score_matrix, key=lambda x: score_matrix[x]['NLPS']
                if not np.isnan(score_matrix[x]['NLPS']) else -1, reverse=True):
    s = score_matrix[m]
    def fmt(v): return f"{v:.3f}" if v is not None and not np.isnan(v) else ' N/A '
    print(f"{m:22s}  {fmt(s['NLC'])}  {fmt(s['BPPI'])}  {fmt(s['TER'])}  {fmt(s['NSS'])}  {fmt(s['NLPS'])}")


## 5. Key claims checklist

Before submitting to NeurIPS, verify each of these against the final score table:

- [ ] **Sensitivity:** At least one pair of methods has NLPS difference > 0.1 (protocol is discriminative)
- [ ] **BPPI vs NLC tradeoff:** At least one method has NLC > 0.6 and BPPI < 0.3 (tradeoff is non-trivial)
- [ ] **NSS hypothesis:** Ancilla tracing NSS > Data re-uploading NSS (open-system noise robustness)
- [ ] **Hybrid upper bound:** Hybrid HQNN NLPS ≥ best purely quantum method (sanity check)
- [ ] **Reproducibility:** All results stable across 3 random seeds (run notebooks with seeds 0, 42, 99)